# ETL — Трансформация mock_data → Звёздная схема (PostgreSQL)

**Шаги:**
1. Читаем таблицу `mock_data` из PostgreSQL через JDBC
2. Строим таблицы измерений и заполняем суррогатные ключи через `row_number()`
3. Строим таблицу фактов `fact_sales` через join-ы по натуральным ключам
4. Записываем все таблицы обратно в PostgreSQL (TRUNCATE + INSERT)

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. SparkSession и подключение к PostgreSQL

In [18]:
spark = (
    SparkSession.builder
    .appName("ETL Star Schema")
    .master("local[*]")
    .config("spark.jars", "/opt/jars/postgresql-42.7.3.jar")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.1.1


In [19]:
PG_URL   = "jdbc:postgresql://postgres:5432/postgres-petshop"
PG_PROPS = {
    "user":     "postgres",
    "password": "postgres",
    "driver":   "org.postgresql.Driver",
}

def read_pg(table: str):
    return spark.read.jdbc(PG_URL, table, properties=PG_PROPS)

def write_pg(df, table: str):
    """TRUNCATE + INSERT через mode=overwrite + truncate=true.
    count() вызывается ДО write, чтобы не материализовывать DataFrame дважды.
    """
    n = df.count()
    df.write.jdbc(
        PG_URL, table, mode="overwrite",
        properties={**PG_PROPS, "truncate": "true"},
    )
    print(f"  ✓ {table}: {n} строк записано")


## 2. Чтение исходных данных

In [20]:
mock = read_pg("public.mock_data")
mock.cache()
print("mock_data rows:", mock.count())
mock.printSchema()

mock_data rows: 10000
root
 |-- id: integer (nullable = true)
 |-- customer_first_name: string (nullable = true)
 |-- customer_last_name: string (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_postal_code: string (nullable = true)
 |-- customer_pet_type: string (nullable = true)
 |-- customer_pet_name: string (nullable = true)
 |-- customer_pet_breed: string (nullable = true)
 |-- seller_first_name: string (nullable = true)
 |-- seller_last_name: string (nullable = true)
 |-- seller_email: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- seller_postal_code: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_price: float (nullable = true)
 |-- product_quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- sale_customer_id: integer (nul

## 3. Window для суррогатных ключей

`row_number()` даёт последовательный INTEGER-ключ от 1.  
Размерностные таблицы небольшие, поэтому сортировка через `monotonically_increasing_id()` как tiebreaker допустима.

In [21]:
# Суррогатные ключи: row_number() по стабильным натуральным ключам
# monotonically_increasing_id() — недетерминирован, каждый запуск даёт разные ключи.
w_loc  = Window.orderBy(
    F.col('country').asc_nulls_last(),
    F.col('state').asc_nulls_last(),
    F.col('city').asc_nulls_last(),
    F.col('postal_code').asc_nulls_last(),
)
w_cat  = Window.orderBy('category_name')
w_pet  = Window.orderBy('pet_type', 'pet_category', 'pet_breed')
w_cust = Window.orderBy('customer_email')
w_sell = Window.orderBy('seller_email')
w_stor = Window.orderBy('store_name', 'email')
w_supp = Window.orderBy('supplier_name', 'email')
w_prod = Window.orderBy('product_name', 'brand', 'color', 'size', 'material')
w_fact = Window.orderBy(F.col('id'))  # id — уникальный номер строки mock_data


## 4. dim_location

Объединяем локации из четырёх источников (покупатели, продавцы, магазины, поставщики).  
Каждый источник заполняет только свои поля:

In [22]:
loc_customer = (
    mock
    .filter(F.col('customer_country').isNotNull())
    .select(
        F.col('customer_country').alias('country'),
        F.lit(None).cast('string').alias('state'),
        F.lit(None).cast('string').alias('city'),
        F.col('customer_postal_code').alias('postal_code'),
    )
)

loc_seller = (
    mock
    .filter(F.col('seller_country').isNotNull())
    .select(
        F.col('seller_country').alias('country'),
        F.lit(None).cast('string').alias('state'),
        F.lit(None).cast('string').alias('city'),
        F.col('seller_postal_code').alias('postal_code'),
    )
)

loc_store = (
    mock
    .filter(F.col('store_country').isNotNull())
    .select(
        F.col('store_country').alias('country'),
        F.col('store_state').alias('state'),
        F.col('store_city').alias('city'),
        F.lit(None).cast('string').alias('postal_code'),
    )
)

loc_supplier = (
    mock
    .filter(F.col('supplier_country').isNotNull())
    .select(
        F.col('supplier_country').alias('country'),
        F.lit(None).cast('string').alias('state'),
        F.col('supplier_city').alias('city'),
        F.lit(None).cast('string').alias('postal_code'),
    )
)

dim_location = (
    loc_customer.union(loc_seller).union(loc_store).union(loc_supplier)
    .distinct()
    .withColumn('location_id', F.row_number().over(w_loc))  # стабильный ключ
    .select('location_id', 'country', 'state', 'city', 'postal_code')
)
dim_location.cache()
print('dim_location rows:', dim_location.count())


dim_location rows: 26733


## 5. dim_category

In [23]:
dim_category = (
    mock
    .filter(F.col("product_category").isNotNull())
    .select(F.col("product_category").alias("category_name"))
    .distinct()
    .withColumn("category_id", F.row_number().over(w_cat))
    .select("category_id", "category_name")
)
dim_category.cache()
print("dim_category rows:", dim_category.count())

dim_category rows: 3


## 6. dim_pet

In [24]:
dim_pet = (
    mock
    .filter(F.col("customer_pet_type").isNotNull())
    .select(
        F.col("customer_pet_type").alias("pet_type"),
        F.col("pet_category"),
        F.col("customer_pet_breed").alias("pet_breed"),
    )
    .distinct()
    .withColumn("pet_id", F.row_number().over(w_pet))
    .select("pet_id", "pet_type", "pet_category", "pet_breed")
)
dim_pet.cache()
print("dim_pet rows:", dim_pet.count())

dim_pet rows: 45


## 7. dim_customer

Дедупликация по `customer_email`.  
Локация клиентов: `city IS NULL` (нет города — только страна и postal_code).

In [25]:
cust_raw = (
    mock
    .filter(F.col('customer_email').isNotNull())
    .select(
        'customer_email',
        F.col('customer_first_name').alias('first_name'),
        F.col('customer_last_name').alias('last_name'),
        F.col('customer_age').alias('age'),
        'customer_country',
        'customer_postal_code',
        'customer_pet_type',
        'pet_category',
        'customer_pet_breed',
        F.col('customer_pet_name').alias('pet_name'),
    )
    .dropDuplicates(['customer_email'])
)

# Локации без города (customer / seller): city IS NULL
# ВАЖНО: postal_code может быть NULL (пустые поля CSV → NULL в PostgreSQL).
# Используем eqNullSafe (<=>), чтобы NULL <=> NULL = True.
loc_no_city = (
    dim_location
    .filter(F.col('city').isNull())
    .select(
        F.col('location_id').alias('cust_loc_id'),
        F.col('country').alias('loc_country'),
        F.col('postal_code').alias('loc_postal'),
    )
)

pet_lookup = dim_pet.select(
    'pet_id',
    F.col('pet_type').alias('j_pet_type'),
    F.col('pet_category').alias('j_pet_cat'),
    F.col('pet_breed').alias('j_pet_breed'),
)

dim_customer = (
    cust_raw
    .join(
        loc_no_city,
        (F.col('customer_country') == F.col('loc_country')) &
        F.col('customer_postal_code').eqNullSafe(F.col('loc_postal')),
        'left',
    )
    .join(
        pet_lookup,
        (F.col('customer_pet_type') == F.col('j_pet_type')) &
        (F.col('pet_category') == F.col('j_pet_cat')) &
        (F.col('customer_pet_breed') == F.col('j_pet_breed')),
        'left',
    )
    .withColumn('customer_id', F.row_number().over(w_cust))
    .select(
        'customer_id',
        'first_name',
        'last_name',
        'age',
        F.col('customer_email').alias('email'),
        F.col('cust_loc_id').alias('location_id'),
        'pet_id',
        'pet_name',
    )
)
dim_customer.cache()
print('dim_customer rows:', dim_customer.count())


dim_customer rows: 10000


## 8. dim_seller

In [26]:
seller_raw = (
    mock
    .filter(F.col('seller_email').isNotNull())
    .select(
        'seller_email',
        F.col('seller_first_name').alias('first_name'),
        F.col('seller_last_name').alias('last_name'),
        'seller_country',
        'seller_postal_code',
    )
    .dropDuplicates(['seller_email'])
)

dim_seller = (
    seller_raw
    .join(
        loc_no_city.withColumnRenamed('cust_loc_id', 'sell_loc_id'),
        (F.col('seller_country') == F.col('loc_country')) &
        F.col('seller_postal_code').eqNullSafe(F.col('loc_postal')),
        'left',
    )
    .withColumn('seller_id', F.row_number().over(w_sell))
    .select(
        'seller_id',
        'first_name',
        'last_name',
        F.col('seller_email').alias('email'),
        F.col('sell_loc_id').alias('location_id'),
    )
)
dim_seller.cache()
print('dim_seller rows:', dim_seller.count())


dim_seller rows: 10000


## 9. dim_store

Локации магазинов: страна + штат + город (есть все три поля, `postal_code IS NULL`).

In [27]:
store_raw = (
    mock
    .filter(F.col('store_name').isNotNull())
    .select(
        'store_name',
        F.col('store_location').alias('location_address'),
        'store_country',
        'store_state',
        'store_city',
        F.col('store_phone').alias('phone'),
        F.col('store_email').alias('email'),
    )
    .dropDuplicates(['store_name', 'email'])
)

# Локации магазинов: country + state (может быть NULL) + city, без postal_code.
# Фильтр: postal_code IS NULL AND city IS NOT NULL.
# state может быть NULL (83.9% записей), поэтому используем eqNullSafe для state.
loc_store_lookup = (
    dim_location
    .filter(F.col('postal_code').isNull() & F.col('city').isNotNull())
    .select(
        F.col('location_id').alias('store_loc_id'),
        F.col('country').alias('st_country'),
        F.col('state').alias('st_state'),
        F.col('city').alias('st_city'),
    )
)

dim_store = (
    store_raw
    .join(
        loc_store_lookup,
        (F.col('store_country') == F.col('st_country')) &
        F.col('store_state').eqNullSafe(F.col('st_state')) &
        (F.col('store_city')    == F.col('st_city')),
        'left',
    )
    .withColumn('store_id', F.row_number().over(w_stor))
    .select(
        'store_id',
        'store_name',
        'location_address',
        F.col('store_loc_id').alias('location_id'),
        'phone',
        'email',
    )
)
dim_store.cache()
print('dim_store rows:', dim_store.count())


dim_store rows: 10000


## 10. dim_supplier

Локации поставщиков: страна + город (`state IS NULL`, `postal_code IS NULL`).

In [28]:
supplier_raw = (
    mock
    .filter(F.col("supplier_name").isNotNull())
    .select(
        "supplier_name",
        F.col("supplier_contact").alias("contact_name"),
        F.col("supplier_email").alias("email"),
        F.col("supplier_phone").alias("phone"),
        F.col("supplier_address").alias("location_address"),
        "supplier_country",
        "supplier_city",
    )
    .dropDuplicates(["supplier_name", "email"])
)

loc_supplier_lookup = (
    dim_location
    .filter(
        F.col("state").isNull() &
        F.col("postal_code").isNull() &
        F.col("city").isNotNull()
    )
    .select(
        F.col("location_id").alias("sup_loc_id"),
        F.col("country").alias("sup_country"),
        F.col("city").alias("sup_city"),
    )
)

dim_supplier = (
    supplier_raw
    .join(
        loc_supplier_lookup,
        (F.col("supplier_country") == F.col("sup_country")) &
        (F.col("supplier_city")    == F.col("sup_city")),
        "left",
    )
    .withColumn("supplier_id", F.row_number().over(w_supp))
    .select(
        "supplier_id",
        "supplier_name",
        "contact_name",
        "email",
        "phone",
        "location_address",
        F.col("sup_loc_id").alias("location_id"),
    )
)
dim_supplier.cache()
print("dim_supplier rows:", dim_supplier.count())

dim_supplier rows: 10000


## 11. dim_product

Дедупликация по (product_name, brand, color, size, material) — уникальный SKU.  
Даты парсим из строки `MM/dd/yyyy` → `DateType`.

In [29]:
product_raw = (
    mock
    .filter(F.col("product_name").isNotNull())
    .select(
        "product_name",
        "product_category",
        F.col("supplier_name").alias("src_sup_name"),
        F.col("supplier_email").alias("src_sup_email"),
        F.col("product_price").alias("price"),
        F.col("product_weight").alias("weight"),
        F.col("product_color").alias("color"),
        F.col("product_size").alias("size"),
        F.col("product_brand").alias("brand"),
        F.col("product_material").alias("material"),
        F.col("product_description").alias("description"),
        F.col("product_rating").alias("rating"),
        F.col("product_reviews").alias("reviews"),
        F.when(
            F.col("product_release_date") != "",
            F.col("product_release_date")
        ).alias("raw_release_date"),
        F.when(
            F.col("product_expiry_date") != "",
            F.col("product_expiry_date")
        ).alias("raw_expiry_date"),
    )
    .dropDuplicates(["product_name", "brand", "color", "size", "material"])
)

cat_lookup = dim_category.select(
    "category_id",
    F.col("category_name").alias("j_cat_name"),
)

sup_lookup = dim_supplier.select(
    "supplier_id",
    F.col("supplier_name").alias("j_sup_name"),
    F.col("email").alias("j_sup_email"),
)

dim_product = (
    product_raw
    .join(cat_lookup, F.col("product_category") == F.col("j_cat_name"), "left")
    .join(
        sup_lookup,
        (F.col("src_sup_name")  == F.col("j_sup_name")) &
        (F.col("src_sup_email") == F.col("j_sup_email")),
        "left",
    )
    .withColumn("product_id", F.row_number().over(w_prod))
    .withColumn("release_date", F.expr("try_to_date(raw_release_date, 'M/d/yyyy')"))
    .withColumn("expiry_date",  F.expr("try_to_date(raw_expiry_date,  'M/d/yyyy')"))
    .select(
        "product_id", "product_name", "category_id", "supplier_id",
        "price", "weight", "color", "size", "brand", "material",
        "description", "rating", "reviews", "release_date", "expiry_date",
    )
)
dim_product.cache()
print("dim_product rows:", dim_product.count())

dim_product rows: 9920


## 12. fact_sales

Каждая строка mock_data — одна транзакция продажи.  
Связываем с размерностями через натуральные ключи.

In [30]:
cust_lookup = dim_customer.select(
    "customer_id",
    F.col("email").alias("cust_email_lk"),
)

sell_lookup = dim_seller.select(
    "seller_id",
    F.col("email").alias("sell_email_lk"),
)

prod_lookup = dim_product.select(
    "product_id",
    F.col("product_name").alias("p_name"),
    F.col("brand").alias("p_brand"),
    F.col("color").alias("p_color"),
    F.col("size").alias("p_size"),
    F.col("material").alias("p_material"),
)

store_lookup = dim_store.select(
    "store_id",
    F.col("store_name").alias("st_name"),
    F.col("email").alias("st_email"),
)

fact_sales = (
    mock
    .join(cust_lookup,  mock.customer_email == F.col("cust_email_lk"), "left")
    .join(sell_lookup,  mock.seller_email   == F.col("sell_email_lk"), "left")
    .join(
        prod_lookup,
        (mock.product_name     == F.col("p_name"))     &
        (mock.product_brand    == F.col("p_brand"))    &
        (mock.product_color    == F.col("p_color"))    &
        (mock.product_size     == F.col("p_size"))     &
        (mock.product_material == F.col("p_material")),
        "left",
    )
    .join(
        store_lookup,
        (mock.store_name  == F.col("st_name")) &
        (mock.store_email == F.col("st_email")),
        "left",
    )
    .withColumn("sale_id", F.row_number().over(w_fact))
    .withColumn(
        "sale_date",
        F.expr("try_to_date(CASE WHEN sale_date != '' THEN sale_date END, 'M/d/yyyy')"),
    )
    .select(
        "sale_id",
        "sale_date",
        "customer_id",
        "seller_id",
        "product_id",
        "store_id",
        F.col("sale_quantity").alias("quantity"),
        F.col("sale_total_price").alias("total_price"),
    )
)
fact_sales.cache()
print("fact_sales rows:", fact_sales.count())

fact_sales rows: 10000


## 13. Запись в PostgreSQL

In [31]:
print("Записываем размерности...")
write_pg(dim_location, "public.dim_location")
write_pg(dim_category, "public.dim_category")
write_pg(dim_pet,      "public.dim_pet")
write_pg(dim_customer, "public.dim_customer")
write_pg(dim_seller,   "public.dim_seller")
write_pg(dim_store,    "public.dim_store")
write_pg(dim_supplier, "public.dim_supplier")
write_pg(dim_product,  "public.dim_product")
print("\nЗаписываем таблицу фактов...")
write_pg(fact_sales,   "public.fact_sales")
print("\nГотово!")

Записываем размерности...
  ✓ public.dim_location: 26733 строк записано
  ✓ public.dim_category: 3 строк записано
  ✓ public.dim_pet: 45 строк записано
  ✓ public.dim_customer: 10000 строк записано
  ✓ public.dim_seller: 10000 строк записано
  ✓ public.dim_store: 10000 строк записано
  ✓ public.dim_supplier: 10000 строк записано
  ✓ public.dim_product: 9920 строк записано

Записываем таблицу фактов...
  ✓ public.fact_sales: 10000 строк записано

Готово!


## 14. Верификация

In [32]:
checks = [
    ("mock_data",    "public.mock_data"),
    ("dim_location", "public.dim_location"),
    ("dim_category", "public.dim_category"),
    ("dim_pet",      "public.dim_pet"),
    ("dim_customer", "public.dim_customer"),
    ("dim_seller",   "public.dim_seller"),
    ("dim_store",    "public.dim_store"),
    ("dim_supplier", "public.dim_supplier"),
    ("dim_product",  "public.dim_product"),
    ("fact_sales",   "public.fact_sales"),
]

print(f"{'Таблица':<20} {'Строк':>8}")
print("-" * 30)
for name, tbl in checks:
    n = read_pg(tbl).count()
    print(f"{name:<20} {n:>8}")

Таблица                 Строк
------------------------------
mock_data               10000
dim_location            26733
dim_category                3
dim_pet                    45
dim_customer            10000
dim_seller              10000
dim_store               10000
dim_supplier            10000
dim_product              9920
fact_sales              10000


In [33]:
spark.stop()